# Chapter 04 — Solution architecture patterns · Homework (self-test)

**Goal:** check that you can shape a system end to end, choose the autonomy level, entry point and integration mechanism, and build the resilience pieces (circuit breaker, idempotent tools, a cross-vendor fallback) that keep it running.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 75–100 minutes
- Estimated API cost: **$0** (everything runs offline against real SDK clients with fake HTTP transports)
- Prerequisites: the practice notebook, and the venv from [00-prerequisites](../../00-prerequisites/README.md)

**How to use it**
1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. Part A: answer 8 scenario questions in the `answers` cell, then run the grader.
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. Part C: write a short design answer, then compare it with the rubric.
5. Run the **Scorecard**. Pass mark: **72%** overall (quiz + exercises), the same bar as the Claude exams' scaled 720/1000.

In [ ]:
import hashlib
import json
import time
from typing import Callable, Protocol

import anthropic
import httpx2
import openai

CLAUDE_MODEL, OPENAI_MODEL = "claude-sonnet-5-5", "gpt-6-luna"

# ---------- grading bookkeeping ----------
RESULTS: dict[int, bool] = {}
SCORE: dict[str, int] = {}


def _run_check(n: int, check: Callable[[], None]) -> None:
    # Run one checker; never let it crash the notebook.
    try:
        check()
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    except NotImplementedError:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    except AssertionError as exc:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    except Exception as exc:  # noqa: BLE001 - report, don't crash
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {str(exc)[:200]}")


# ---------- offline vendors: real SDK clients, fake HTTP layer (same as the practice notebook) ----------
def api_error(vendor: str, status: int, kind: str, message: str = "scripted failure") -> dict:
    body = ({"type": "error", "error": {"type": kind, "message": message}} if vendor == "claude"
            else {"error": {"message": message, "type": kind, "code": kind, "param": None}})
    return {"_status": status, "_headers": {"retry-after-ms": "10"}, **body}


def _reply(vendor: str, text: str) -> dict:
    if vendor == "claude":
        return {"id": "msg_fake", "type": "message", "role": "assistant", "model": CLAUDE_MODEL,
                "content": [{"type": "text", "text": text}], "stop_reason": "end_turn", "stop_sequence": None,
                "usage": {"input_tokens": 20, "output_tokens": 5}}
    return {"id": "resp_fake", "object": "response", "created_at": 1790000000, "status": "completed",
            "model": OPENAI_MODEL, "output": [{"type": "message", "id": "m1", "role": "assistant", "status": "completed",
            "content": [{"type": "output_text", "text": text, "annotations": []}]}],
            "usage": {"input_tokens": 20, "input_tokens_details": {"cached_tokens": 0}, "output_tokens": 5,
                      "output_tokens_details": {"reasoning_tokens": 0}, "total_tokens": 25}}


class FakeVendor:
    # script() returns the model's text, or api_error(...). Records request bodies in .requests.
    def __init__(self, vendor: str, script: Callable[[], str | dict], max_retries: int = 0):
        self.vendor, self.script, self.requests = vendor, script, []
        http = httpx2.Client(transport=httpx2.MockTransport(self._handle))
        cls = anthropic.Anthropic if vendor == "claude" else openai.OpenAI
        self.client = cls(api_key="offline", max_retries=max_retries, http_client=http)

    def _handle(self, request: httpx2.Request) -> httpx2.Response:
        self.requests.append(json.loads(request.content))
        out = self.script()
        out = dict(_reply(self.vendor, out) if isinstance(out, str) else out)
        status, headers = out.pop("_status", 200), out.pop("_headers", {})
        return httpx2.Response(status, json=out, headers=headers)


class LLM(Protocol):
    name: str
    def complete(self, system: str, user: str) -> str: ...


class ClaudeLLM:
    def __init__(self, client, model: str = CLAUDE_MODEL):
        self.client, self.model, self.name = client, model, "claude"

    def complete(self, system: str, user: str) -> str:
        r = self.client.messages.create(model=self.model, max_tokens=256, system=system,
                                        messages=[{"role": "user", "content": user}])
        return "".join(b.text for b in r.content if b.type == "text")


class OpenAILLM:
    def __init__(self, client, model: str = OPENAI_MODEL):
        self.client, self.model, self.name = client, model, "openai"

    def complete(self, system: str, user: str) -> str:
        return self.client.responses.create(model=self.model, instructions=system, input=user,
                                            max_output_tokens=256).output_text


class FakeClock:
    def __init__(self):
        self.t = 0.0

    def __call__(self) -> float:
        return self.t


class SimpleBreaker:
    # The README's breaker (used by Exercise 4). Exercise 2 asks you to build a stricter one.
    def __init__(self, threshold=2, cooldown_s=30.0, clock=time.monotonic):
        self.threshold, self.cooldown_s, self.clock, self.failures, self.opened_at = threshold, cooldown_s, clock, 0, None

    def allow(self) -> bool:
        return self.opened_at is None or self.clock() - self.opened_at >= self.cooldown_s

    def record(self, ok: bool) -> None:
        self.failures = 0 if ok else self.failures + 1
        if ok or self.failures >= self.threshold:
            self.opened_at = None if ok else self.clock()


def pick_integration(other_side_reasons: bool, clients: int, agent_has_shell: bool, cli_exists: bool) -> str:
    # The README's rule (4.5), used by Exercise 1.
    if other_side_reasons:
        return "a2a"
    if agent_has_shell and cli_exists:
        return "cli"
    if clients > 1:
        return "mcp"
    return "api"


print("helpers ready | anthropic", anthropic.__version__, "| openai", openai.__version__)

## Part A — Quiz (8 questions)

Pick one letter per question. The scenarios are original and follow the style of the CCAR-P and OpenAI Academy scoping questions.

**Q1. (End-to-end design)** A legal-tech team launched a contract summarizer three months ago. Partners say quality "feels worse" since a prompt change last month, but nobody can confirm or deny it. The logs hold only request counts and latency. What is the most important thing missing from the architecture?

- A) A larger model, so quality cannot get worse
- B) A longer system prompt with more examples of good summaries
- C) A feedback stage: per-request traces, a labeled sample of outputs that grows into an eval set, and a rule that prompt or model changes ship only when the eval set shows no regression
- D) A thumbs-up/down button for users, and changing the prompt whenever the ratings dip

**Q2. (Autonomy level)** A logistics company wants to auto-answer "where is my parcel?" messages. One tracking-API lookup answers almost all of them, and the reply is a short template. The team proposes an agent on an agent SDK with five tools "for future needs". What do you recommend?

- A) A workflow: one structured-output call extracts the tracking number, code calls the tracking API and fills the template, and anything else goes to the existing support queue
- B) The agent with five tools, because it will handle new kinds of questions later
- C) A coordinator agent with one subagent per carrier
- D) Fine-tuning a model on two years of past replies

**Q3. (Integration)** Claude Code runs in CI and must read pull requests and post review comments. The runner already has the `gh` CLI, authenticated with a token scoped to the repository. A teammate proposes building and hosting an MCP server that wraps the GitHub API for this job. What is the strongest response?

- A) Build the MCP server, because MCP is the standard and is always the better choice
- B) Expose GitHub to the agent through an agent-to-agent (A2A) endpoint
- C) Give the agent raw HTTP access to the GitHub API and the token
- D) Use the `gh` CLI: it is installed, already scoped, costs no tool definitions in context and needs no server to run; add an MCP server only if it brings something the CLI cannot (for example a cross-repository index)

**Q4. (Integration)** A bank's onboarding agent must ask a partner's identity-verification agent to run a KYC check. The check can take hours, the partner's agent sometimes asks for an extra document, and it reports progress along the way. Which mechanism fits?

- A) An MCP tool `run_kyc` that blocks until the check is finished
- B) Agent-to-agent (A2A): discover the partner agent from its Agent Card, send a task, answer its input-required requests, and track the task until it completes or fails
- C) A nightly CSV exchange over SFTP
- D) Read the partner's verification database directly

**Q5. (Resilience)** Every Claude call in an app is wrapped in a decorator that retries 4 more times. The call runs inside a worker that re-runs a failed job up to 2 more times. The SDK keeps its default `max_retries=2`. During a 20-minute `529 overloaded_error` incident, what happens and what is the fix?

- A) At most 3 attempts per job; nothing needs to change
- B) Raise the SDK's `max_retries` to 6 so jobs eventually succeed
- C) Up to 3 × 5 × 3 = 45 attempts per job, all aimed at an overloaded provider; let the SDK own transport retries, remove the decorator, and add a circuit breaker so jobs fail fast (or fall back) while it is open
- D) Switch every call to the async client so the retries run in parallel

**Q6. (Idempotency)** A support agent opens tickets with a `create_ticket` tool. After network timeouts the agent retries, and duplicate tickets appear. A teammate proposes deduplicating on the tool-call id. What is the best design?

- A) Deduplicate on the tool-call id, since every tool call has a unique one
- B) Derive the idempotency key from the business identity (for example the inbound email's message id), store the result with the key, return the stored result on a repeat call, and mark the tool `idempotentHint` if it is exposed over MCP
- C) Add "never create the same ticket twice" to the system prompt
- D) Turn off all retries in the SDK and in the agent

**Q7. (Entry points)** A 300-person consulting firm wants consultants to draft proposals from past proposals and research notes. There is no product to embed the feature in. Security requires SSO and admin controls. What should the architect recommend first?

- A) A custom web app on the Messages API with its own document store
- B) A service built on an agent SDK, with tools for the document store
- C) Claude Code licenses for every consultant
- D) A chat workspace (Claude Enterprise or ChatGPT Enterprise) with projects that hold the reference material, moving to a custom build only if a named requirement such as a custom UI or unattended runs appears

**Q8. (Two vendors)** The CTO wants "full vendor independence": a wrapper that exposes only the features both vendors share, the same prompt on both, and switching vendors by a config flag with no further testing. What is the best critique?

- A) Abstract at the task level (complete, classify, extract with shared schemas and one shared eval set), keep vendor-specific features such as hosted tools, caching and agent runtimes inside the adapters, tune prompts per vendor, and switch only after the eval set passes on the new vendor
- B) The plan is right as stated
- C) Avoid any abstraction and call each SDK directly everywhere in the code
- D) Pick the vendor with the cheapest tokens and never switch

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
}

In [ ]:
ANSWER_HASHES = {'q1': '227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e', 'q2': '3d9c97ee45947882f5eee5cf878c4f33a0a270d8f2a2e1fc259d66cdc95c7b45', 'q3': '14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6', 'q4': '61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa', 'q5': 'e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f', 'q6': '621f4cab9e29d186b9dabeec79a433d2126c30a9e3f5712a1411fc67d73f1e40', 'q7': '45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457', 'q8': 'f98cc196593da1da1a7b4550be81b590cd44e7098497c2034a83c78f8a98f3bc'}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Coding exercises (offline)

Each exercise: read the task, replace the stub, run the checker.

### Exercise 1 — A design-review linter

Write `review_design(design) -> set[str]` that returns the issue codes below (an empty set for a clean design). `design` has these keys (see the example in the checker):

| Code | Raise it when |
|---|---|
| `"no_feedback_loop"` | `design["feedback"]` is missing `traces` **or** `eval_set` (either one is falsy) |
| `"agent_for_known_path"` | `design["autonomy"] == "agent"` and `design["path_known"]` is true |
| `"stacked_retries"` | more than one entry in `design["retry_layers"]` (a dict layer → retry count) has a count above 0 |
| `"wrong_integration"` | any item in `design["integrations"]` has a `mechanism` different from `pick_integration(other_side_reasons, clients, agent_has_shell, cli_exists)` for that item |
| `"non_idempotent_tool"` | any item in `design["tools"]` has `side_effect` true and an `idempotency_key` that is `None` or `"tool_call_id"` |
| `"undefended_decision"` | any item in `design["decisions"]` has an empty or missing `rejected` |

Missing list keys (`integrations`, `tools`, `decisions`) mean empty lists. [Theory: the whole README; 4.5 for the integration rule](README.md#45-integration-mechanisms-mcp-apicli-or-agent-to-agent)

In [ ]:
def review_design(design: dict) -> set[str]:
    """Return the set of issue codes from the table above."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    def integ(mech, reasons=False, clients=1, shell=False, cli=False):
        return {"system": "x", "mechanism": mech, "other_side_reasons": reasons, "clients": clients,
                "agent_has_shell": shell, "cli_exists": cli}

    clean = {
        "autonomy": "workflow", "path_known": True,
        "feedback": {"traces": True, "eval_set": True},
        "retry_layers": {"sdk": 2, "app": 0, "worker": 0},
        "integrations": [integ("api"), integ("mcp", clients=3), integ("cli", shell=True, cli=True),
                         integ("a2a", reasons=True)],
        "tools": [{"name": "lookup", "side_effect": False, "idempotency_key": None},
                  {"name": "refund", "side_effect": True, "idempotency_key": "refund:{order_id}"}],
        "decisions": [{"decision": "shape", "choice": "router", "rejected": "agent"}],
    }
    got = review_design(clean)
    assert got == set(), f"a clean design should give set(), got {got!r}"
    cases = [
        ({"feedback": {"traces": True, "eval_set": False}}, {"no_feedback_loop"}),
        ({"feedback": {}}, {"no_feedback_loop"}),
        ({"autonomy": "agent"}, {"agent_for_known_path"}),
        ({"autonomy": "agent", "path_known": False}, set()),
        ({"retry_layers": {"sdk": 2, "app": 3, "worker": 0}}, {"stacked_retries"}),
        ({"retry_layers": {"sdk": 0, "app": 3}}, set()),
        ({"integrations": [integ("mcp")]}, {"wrong_integration"}),                  # one client: api
        ({"integrations": [integ("mcp", shell=True, cli=True, clients=2)]}, {"wrong_integration"}),  # cli
        ({"integrations": [integ("mcp", reasons=True, clients=5)]}, {"wrong_integration"}),        # a2a
        ({"tools": [{"name": "t", "side_effect": True, "idempotency_key": "tool_call_id"}]}, {"non_idempotent_tool"}),
        ({"tools": [{"name": "t", "side_effect": True, "idempotency_key": None}]}, {"non_idempotent_tool"}),
        ({"decisions": [{"decision": "entry point", "choice": "API"}]}, {"undefended_decision"}),
        ({"decisions": [], "tools": [], "integrations": []}, set()),
    ]
    for change, expected in cases:
        design = {**clean, **change}
        got = review_design(design)
        assert got == expected, f"after changing {change}: expected {expected or 'set()'}, got {got or 'set()'}"
    minimal = {"autonomy": "augmented_llm", "path_known": True, "feedback": {"traces": True, "eval_set": True},
               "retry_layers": {"sdk": 2}}
    assert review_design(minimal) == set(), "missing integrations/tools/decisions keys must count as empty lists"


_run_check(1, check_ex1)

### Exercise 2 — A circuit breaker with a real half-open state

The README's breaker lets *every* caller through once the cool-down has passed. Build a stricter `CircuitBreaker(threshold, cooldown_s, clock)`:

- `state` (a property) returns `"closed"`, `"open"` or `"half_open"`.
- `allow()`: **closed** → `True`. **open** and less than `cooldown_s` since it opened → `False`. **open** and the cool-down has passed → move to **half_open** and return `True` (this caller is the single trial). **half_open** (a trial is in flight) → `False`.
- `record(ok)`: `ok=True` → **closed**, failure count reset. `ok=False` in **half_open** → **open** again, cool-down restarts at the current `clock()`. `ok=False` in **closed** → count it; at `threshold` consecutive failures → **open** (cool-down starts now).

Use only the injected `clock()` for time. [Theory](README.md#46-resilience-patterns)

In [ ]:
class CircuitBreaker:
    def __init__(self, threshold: int = 3, cooldown_s: float = 30.0, clock: Callable[[], float] = time.monotonic):
        # TODO
        raise NotImplementedError

    @property
    def state(self) -> str:
        raise NotImplementedError

    def allow(self) -> bool:
        raise NotImplementedError

    def record(self, ok: bool) -> None:
        raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    clock = FakeClock()
    b = CircuitBreaker(threshold=3, cooldown_s=30, clock=clock)
    assert b.state == "closed" and b.allow(), "a new breaker is closed and allows calls"
    b.record(False); b.record(False)
    assert b.state == "closed", "2 failures with threshold 3: still closed"
    b.record(True); b.record(False); b.record(False)
    assert b.state == "closed", "a success resets the count: failures must be CONSECUTIVE"
    b.record(False)
    assert b.state == "open", "3 consecutive failures must open the breaker"
    clock.t = 29.9
    assert not b.allow(), "open and still cooling down: allow() must be False"
    clock.t = 30.0
    assert b.allow() and b.state == "half_open", "after the cool-down, the first allow() is the trial (half_open)"
    assert not b.allow(), "half_open: only ONE trial may be in flight, the next allow() must be False"
    b.record(False)
    assert b.state == "open", "a failed trial reopens the breaker"
    clock.t = 45.0
    assert not b.allow(), "the cool-down restarts at the failed trial (t=30), so at t=45 it is still open"
    clock.t = 60.0
    assert b.allow() and b.state == "half_open", "second trial at t=60"
    b.record(True)
    assert b.state == "closed" and b.allow(), "a successful trial closes the breaker"
    b.record(False); b.record(False)
    assert b.state == "closed", "after closing, the failure count must start again from 0"


_run_check(2, check_ex2)

### Exercise 3 — Make a side-effecting tool idempotent

Write `make_idempotent(fn, key_fn, store, ignore=("tool_call_id",))`. It returns a function `wrapper(**kwargs)` that:

1. computes `key = key_fn(**kwargs)`;
2. if `key` is in `store`: compares the stored arguments with `kwargs`, **ignoring** the names in `ignore`. Same → return the stored result without calling `fn`. Different → raise `ValueError` (a key reused for a different action is a bug, not a retry);
3. otherwise calls `fn(**kwargs)`. If `fn` raises, store **nothing** and let the exception propagate (a failed attempt must be retryable). If it returns, store `{"args": <kwargs without ignored names>, "result": result}` under `key` and return the result.

[Theory](README.md#46-resilience-patterns)

In [ ]:
def make_idempotent(fn: Callable[..., dict], key_fn: Callable[..., str], store: dict,
                    ignore: tuple[str, ...] = ("tool_call_id",)) -> Callable[..., dict]:
    """Wrap fn so repeats of the same business action return the first result."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    ledger, store, fail_next = [], {}, {"value": False}

    def issue_refund(order_id: str, amount: int, tool_call_id: str) -> dict:
        if fail_next["value"]:
            fail_next["value"] = False
            raise TimeoutError("payment API timed out")
        ledger.append((order_id, amount))
        return {"refund_id": f"rf_{len(ledger)}", "order_id": order_id, "amount": amount}

    refund = make_idempotent(issue_refund, lambda **kw: f"refund:{kw['order_id']}", store)
    r1 = refund(order_id="A-1", amount=4900, tool_call_id="toolu_01")
    r2 = refund(order_id="A-1", amount=4900, tool_call_id="toolu_02")
    assert len(ledger) == 1, f"the same refund with a new tool-call id must not run twice (ledger: {ledger})"
    assert r1 == r2 == {"refund_id": "rf_1", "order_id": "A-1", "amount": 4900}, "a repeat must return the stored result"
    refund(order_id="B-2", amount=1000, tool_call_id="toolu_03")
    assert len(ledger) == 2, "a different business key is a different action and must run"
    try:
        refund(order_id="A-1", amount=9999, tool_call_id="toolu_04")
        raise AssertionError("same key with a different amount must raise ValueError")
    except ValueError:
        pass
    fail_next["value"] = True
    try:
        refund(order_id="C-3", amount=500, tool_call_id="toolu_05")
        raise AssertionError("the TimeoutError from fn must propagate")
    except TimeoutError:
        pass
    assert "refund:C-3" not in store, "a failed attempt must not be stored"
    refund(order_id="C-3", amount=500, tool_call_id="toolu_06")
    assert ledger[-1] == ("C-3", 500), "after a failure, a retry must be able to run the action"
    assert store["refund:C-3"]["args"] == {"order_id": "C-3", "amount": 500}, "stored args must exclude ignored names"


_run_check(3, check_ex3)

### Exercise 4 — A fallback chain that knows which errors to fall back on

Write `call_with_fallback(providers, breakers, system, user) -> dict`. `providers` is a list of `LLM` adapters (here `ClaudeLLM` / `OpenAILLM` on `FakeVendor` clients with `max_retries=0`); `breakers` maps `llm.name` to a `SimpleBreaker` (helpers cell). For each provider in order:

- breaker does not `allow()` → record the attempt `(name, "skipped")` and move on;
- call `llm.complete(system, user)`. Success → `record(True)`, attempt `(name, "ok")`, return `{"provider": name, "text": text, "attempts": attempts}`;
- **transient** failure → `record(False)`, attempt `(name, "transient_error")`, move on. Transient means `anthropic.APIConnectionError` / `openai.APIConnectionError` (timeouts included), or an `APIStatusError` from either SDK with `status_code` 429 or ≥ 500;
- **any other** `APIStatusError` (400, 401, 403, 404, 413, ...) → re-raise immediately, **without** recording a failure on the breaker: an invalid request is your bug, it would fail on the fallback too, and it must not open the breaker for everyone else.

If no provider succeeds, return `{"provider": None, "text": None, "attempts": attempts, "route": "human_queue"}`. [Theory](README.md#46-resilience-patterns)

In [ ]:
def call_with_fallback(providers: list, breakers: dict, system: str, user: str) -> dict:
    """Try providers in order; fall back only on transient errors."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    clock = FakeClock()
    claude_mode = {"value": "down"}

    def claude_script():
        if claude_mode["value"] == "down":
            return api_error("claude", 529, "overloaded_error")
        if claude_mode["value"] == "rate_limited":
            return api_error("claude", 429, "rate_limit_error")
        if claude_mode["value"] == "bad_request":
            return api_error("claude", 400, "invalid_request_error", "messages: field required")
        return "claude answer"

    openai_mode = {"value": "up"}
    fc = FakeVendor("claude", claude_script)
    fo = FakeVendor("openai", lambda: "openai answer" if openai_mode["value"] == "up"
                    else api_error("openai", 503, "server_error"))
    providers = [ClaudeLLM(fc.client), OpenAILLM(fo.client)]
    breakers = {"claude": SimpleBreaker(threshold=2, clock=clock), "openai": SimpleBreaker(threshold=2, clock=clock)}

    r = call_with_fallback(providers, breakers, "sys", "hi")
    assert isinstance(r, dict) and r.get("provider") == "openai" and r.get("text") == "openai answer", \
        f"Claude 529 -> fall back to OpenAI; got {r!r}"
    assert r["attempts"] == [("claude", "transient_error"), ("openai", "ok")], f"attempts wrong: {r['attempts']}"
    call_with_fallback(providers, breakers, "sys", "hi")
    n_claude = len(fc.requests)
    r = call_with_fallback(providers, breakers, "sys", "hi")
    assert len(fc.requests) == n_claude and r["attempts"][0] == ("claude", "skipped"), \
        "after 2 transient failures the Claude breaker is open: skip Claude without sending a request"
    clock.t += 31
    claude_mode["value"] = "rate_limited"
    r = call_with_fallback(providers, breakers, "sys", "hi")
    assert r["attempts"][0] == ("claude", "transient_error") and r["provider"] == "openai", "429 is transient"
    clock.t += 31
    claude_mode["value"] = "up"
    r = call_with_fallback(providers, breakers, "sys", "hi")
    assert r["provider"] == "claude" and r["attempts"] == [("claude", "ok")], "Claude recovered: it must serve again"
    claude_mode["value"] = "bad_request"
    n_openai = len(fo.requests)
    try:
        call_with_fallback(providers, breakers, "sys", "hi")
        raise AssertionError("a 400 must be re-raised, not swallowed or retried on OpenAI")
    except anthropic.BadRequestError:
        pass
    assert len(fo.requests) == n_openai, "a 400 from Claude must NOT fall back to OpenAI"
    assert breakers["claude"].failures == 0 and breakers["claude"].opened_at is None, \
        "a 400 must not count as a failure on the breaker"
    claude_mode["value"], openai_mode["value"] = "down", "down"
    r = call_with_fallback(providers, breakers, "sys", "hi")
    assert r.get("provider") is None and r.get("route") == "human_queue", f"all down -> human_queue, got {r!r}"


_run_check(4, check_ex4)

## Part C — Architecture scenario

**City permits office.** A city government wants three things:

1. **Residents** ask questions about permit rules on the city website. The rules are 600 pages of regulations, updated monthly. The service must stay up during a model-vendor outage.
2. **Staff** want each incoming permit application (about 3,000 a month: a PDF plus a web form) **pre-checked for completeness**, with the result written into the permit system. The mail gateway sometimes delivers an application twice.
3. Two integrations: the state land registry offers an **agent** that answers ownership questions and sometimes asks for a parcel id; the city's **GIS system** has a REST API already used by three internal apps, and the new assistant would be the fourth client.

Procurement wants a path that does not lock the city into one model vendor. Write a one-page design (as in README [4.8](README.md#48-writing-it-down-the-architecture-one-pager)): the shape and autonomy level for each use case with a rejected alternative, the entry point, the integration mechanism for each system, the four stages, the resilience plan, the feedback loop, and how Claude and OpenAI fit.

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **Resident Q&A = grounded Q&A blueprint (augmented LLM call with retrieval)**, not an agent: answers only from retrieved regulation sections, with citations, and a "not covered, contact the office" refusal when nothing relevant is found. Re-index on each monthly update. Rejected alternative: an agent with web browsing (cost, variance, no extra capability). Entry point: the model API behind the city's own website (a public, unauthenticated audience rules out a staff chat workspace).
- **Application pre-check = workflow (extraction pipeline)**: classify the application type → extract required fields into a strict schema with nullable fields → completeness rules **in code** → one repair attempt → incomplete or unclear cases go to staff. It is not urgent, so it can run in batches at lower cost. Rejected alternative: a per-application agent.
- **Idempotent write** into the permit system, keyed on the business identity (application id or form submission id, not the delivery or tool-call id); duplicates return the stored result.
- **Integrations:** the land registry is a reasoning agent that asks follow-up questions → **A2A** (Agent Card, tasks, handle input-required, timeouts). GIS already has several consumers and is gaining another → an **MCP server** (or reuse of an existing one) with read-only tools and scoped auth; a direct API call is also acceptable if the answer says why. No raw database access.
- **Four stages** stated for each flow: input (file type and size limits, injection screening of uploaded PDFs, PII handling for applicant data), processing, output (schema and citation checks, staff approval before anything is sent to an applicant), feedback.
- **Resilience:** the SDK owns transport retries (no stacked loops), timeouts per call, circuit breakers per vendor, a **cross-vendor fallback for resident Q&A** behind a task-level interface, with its own prompt tuning and eval pass; a static "service degraded" answer plus office contact details as the last fallback.
- **Feedback:** traces per request; staff corrections and sampled resident questions labeled into eval sets (one per flow, with per-segment coverage); prompt, model or vendor changes gated on those eval sets; resident feedback reviewed, not used directly as a gate.
- **Claude vs OpenAI:** both reachable through the same interface; vendor-specific features (for example OpenAI's hosted file search vs bringing your own retriever on Claude) live in the adapters; the vendor choice is made on the shared eval set and on cost and data controls, and recorded with its rejected alternative.
</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_q, n_ex = 8, 4
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (n_q + n_ex)
print(f"Quiz:       {quiz}/{n_q} ({quiz / n_q:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS: on to the next chapter" if overall >= 0.72 else "RETRY: reread the README sections behind your ✗ items, then try again")

Back to the [chapter README](README.md) · [Applied AI architect track](../README.md)

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **C.** Without traces, labeled cases and a release gate, nobody can measure quality, so the "feels worse" question cannot be answered and the next change is a guess too. A changes the model without measurement. B changes the prompt without measurement. D collects a signal but has no labeled cases and no gate; reacting to rating dips is still guessing.

**Q2:** **A.** The path is known: extract, look up, fill the template. A workflow is cheaper, predictable and testable per step, with a clear route for everything else. B adds autonomy and cost for needs nobody has named. C is multi-agent over-engineering. D bakes yesterday's replies into a model and does not call the tracking API at all.

**Q3:** **D.** For a coding agent with a shell, an installed and scoped CLI is the cheapest integration. A treats a standard as a reason on its own and adds a server to run and secure. B is for systems that reason; GitHub's API does not. C hands the model a broader credential and raw HTTP for no gain.

**Q4:** **B.** The other side is an agent that runs long, asks for input and reports progress: the agent-to-agent case. A hides the questions inside one blocking call and has no good way to wait hours. C and D are not agent integrations, and D bypasses the partner's interface and controls.

**Q5:** **C.** Nested retries multiply: 3 SDK attempts × 5 decorator attempts × 3 worker runs = 45, all aimed at a provider that is already overloaded. One retry owner plus a breaker (and a fallback) is the fix. A miscounts. B makes the retry storm larger. D sends the same storm faster.

**Q6:** **B.** A retry or a resumed run produces a new tool-call id, so A lets the duplicate through. The key must come from the business action, and the stored result is returned on repeats; the MCP hint helps clients but the tool enforces it. C is guidance where a guarantee is needed. D removes useful transport retries and still does not stop a re-planned call.

**Q7:** **D.** Start at the highest-level entry point that meets the need: interactive drafting with reference material, SSO and admin controls is what an enterprise chat workspace provides. A and B build and operate a product nobody asked for. C is a coding agent, the wrong tool for proposal writing.

**Q8:** **A.** Abstract the task, not the lowest common denominator, keep vendor-specific strengths in the adapters, and treat a vendor switch like any other change: gated by the shared eval set. B assumes prompts transfer and skips testing. C leaks vendor shapes everywhere and makes a switch expensive. D optimizes one number and gives up the fallback.
</details>

<details><summary>Exercise 1 — A design-review linter: reference solution</summary>

```python
def review_design(design: dict) -> set[str]:
    issues = set()
    fb = design.get("feedback", {})
    if not (fb.get("traces") and fb.get("eval_set")):
        issues.add("no_feedback_loop")
    if design.get("autonomy") == "agent" and design.get("path_known"):
        issues.add("agent_for_known_path")
    if sum(1 for n in design.get("retry_layers", {}).values() if n > 0) > 1:
        issues.add("stacked_retries")
    for i in design.get("integrations", []):
        expected = pick_integration(i["other_side_reasons"], i["clients"], i["agent_has_shell"], i["cli_exists"])
        if i["mechanism"] != expected:
            issues.add("wrong_integration")
    for t in design.get("tools", []):
        if t.get("side_effect") and t.get("idempotency_key") in (None, "tool_call_id"):
            issues.add("non_idempotent_tool")
    if any(not d.get("rejected") for d in design.get("decisions", [])):
        issues.add("undefended_decision")
    return issues
```
</details>

<details><summary>Exercise 2 — A circuit breaker with a real half-open state: reference solution</summary>

```python
class CircuitBreaker:
    def __init__(self, threshold: int = 3, cooldown_s: float = 30.0, clock: Callable[[], float] = time.monotonic):
        self.threshold, self.cooldown_s, self.clock = threshold, cooldown_s, clock
        self._state, self.failures, self.opened_at = "closed", 0, None

    @property
    def state(self) -> str:
        return self._state

    def allow(self) -> bool:
        if self._state == "closed":
            return True
        if self._state == "open" and self.clock() - self.opened_at >= self.cooldown_s:
            self._state = "half_open"            # this caller is the single trial
            return True
        return False                             # open and cooling down, or a trial is in flight

    def record(self, ok: bool) -> None:
        if ok:
            self._state, self.failures, self.opened_at = "closed", 0, None
            return
        if self._state == "half_open":
            self._state, self.opened_at = "open", self.clock()
            return
        self.failures += 1
        if self.failures >= self.threshold:
            self._state, self.opened_at = "open", self.clock()
```
</details>

<details><summary>Exercise 3 — Make a side-effecting tool idempotent: reference solution</summary>

```python
def make_idempotent(fn: Callable[..., dict], key_fn: Callable[..., str], store: dict,
                    ignore: tuple[str, ...] = ("tool_call_id",)) -> Callable[..., dict]:
    def wrapper(**kwargs):
        key = key_fn(**kwargs)
        args = {k: v for k, v in kwargs.items() if k not in ignore}
        if key in store:
            if store[key]["args"] != args:
                raise ValueError(f"idempotency key {key!r} reused with different arguments")
            return store[key]["result"]
        result = fn(**kwargs)                   # if this raises, nothing is stored
        store[key] = {"args": args, "result": result}
        return result
    return wrapper
```
</details>

<details><summary>Exercise 4 — A fallback chain that knows which errors to fall back on: reference solution</summary>

```python
def _is_transient(exc: Exception) -> bool:
    if isinstance(exc, (anthropic.APIConnectionError, openai.APIConnectionError)):
        return True
    if isinstance(exc, (anthropic.APIStatusError, openai.APIStatusError)):
        return exc.status_code == 429 or exc.status_code >= 500
    return False


def call_with_fallback(providers: list, breakers: dict, system: str, user: str) -> dict:
    attempts = []
    for llm in providers:
        breaker = breakers[llm.name]
        if not breaker.allow():
            attempts.append((llm.name, "skipped"))
            continue
        try:
            text = llm.complete(system, user)
        except (anthropic.APIError, openai.APIError) as exc:
            if not _is_transient(exc):
                raise                              # our bug: no fallback, no breaker trip
            breaker.record(False)
            attempts.append((llm.name, "transient_error"))
            continue
        breaker.record(True)
        attempts.append((llm.name, "ok"))
        return {"provider": llm.name, "text": text, "attempts": attempts}
    return {"provider": None, "text": None, "attempts": attempts, "route": "human_queue"}
```
</details>